# Vocabulary Tools for Agentic Systems
> Extension tools for working with semantic vocabularies in CogitareLink

In [ ]:
#| default_exp cli.vocab_tools

In [ ]:
#| export
from __future__ import annotations
import json
import time
from typing import Dict, List, Any, Optional, Union, Tuple, Callable
from pathlib import Path

from cogitarelink.cli.cli import Agent, ToolRegistry
from cogitarelink.core.debug import get_logger
from cogitarelink.core.cache import InMemoryCache

In [ ]:
#| exporti
__all__ = ['VocabToolAgent', 'registry_tools', 'context_tools', 'retrieval_tools']

# Building Vocabulary Tools

In this notebook, we'll create specialized tools for working with semantic vocabularies, extending the agent infrastructure from the main CLI module. These tools will allow agents to:

1. **Explore and manage vocabulary registries** - Browse, search, and manipulate entries in the registry
2. **Compose and analyze contexts** - Create, merge, and analyze JSON-LD contexts
3. **Retrieve and process vocabulary resources** - Fetch, convert, and validate vocabulary data
4. **Integrate with Wikidata** - Search, align, and enrich data using Wikidata

We'll organize these tools into logical groups and create a specialized agent class that comes pre-configured with all the vocabulary tools.

# 1. Registry Tools

First, let's create tools for exploring and managing the vocabulary registry. These tools will wrap the functionality in `vocab/registry.py`.

In [ ]:
#| export
def registry_tools(registry=None) -> Dict[str, Callable]:
    """Create a set of tools for working with the vocabulary registry.
    
    Returns:
        Dict mapping tool names to tool functions
    """
    from cogitarelink.vocab.registry import registry as default_registry
    reg = registry or default_registry
    
    tools = {}
    
    def explore_registry(filter_tags: List[str] = None) -> Dict[str, Dict]:
        """
        Explore available vocabularies in the registry with optional filtering.
        
        Parameters:
            filter_tags: Optional tags to filter vocabularies
            
        Returns:
            Dictionary of vocabulary entries matching the filter
        """
        result = {}
        for prefix, entry in reg._v.items():
            if filter_tags and not any(tag in entry.tags for tag in filter_tags):
                continue
            result[prefix] = {
                "prefix": entry.prefix,
                "uri": entry.uris.get("primary"),
                "versions": entry.versions.model_dump(),  # Updated from dict() to model_dump() for pydantic v2
                "features": list(entry.features)
            }
        return result
    
    def get_vocabulary_info(prefix: str) -> Dict[str, Any]:
        """
        Get detailed information about a specific vocabulary.
        
        Parameters:
            prefix: Vocabulary prefix
            
        Returns:
            Complete vocabulary entry information
        """
        if prefix not in reg._v:
            return {"success": False, "error": f"Vocabulary '{prefix}' not found"}
            
        entry = reg[prefix]
        return {
            "success": True,
            "prefix": entry.prefix,
            "uris": {k: str(v) for k, v in entry.uris.items()},
            "context": {
                "url": str(entry.context.url) if entry.context.url else None,
                "inline": entry.context.inline is not None,
                "derives_from": str(entry.context.derives_from) if entry.context.derives_from else None,
                "sha256": entry.context.sha256
            },
            "versions": entry.versions.model_dump(),  # Updated from dict() to model_dump() for pydantic v2
            "features": list(entry.features),
            "tags": list(entry.tags),
            "strategy_defaults": entry.strategy_defaults,
            "meta": entry.meta
        }
    
    def add_temp_vocabulary(prefix: str, uri: str, context_url: str = None, 
                           inline_context: Dict = None, ttl_url: str = None) -> Dict:
        """
        Add a temporary vocabulary to the registry for the current session.
        
        Parameters:
            prefix: Short prefix for the vocabulary
            uri: Primary URI for the vocabulary
            context_url: Optional JSON-LD context URL
            inline_context: Optional inline context definition
            ttl_url: Optional TTL file URL to derive context from
            
        Returns:
            Dictionary with operation result
        """
        from cogitarelink.vocab.registry import VocabEntry, ContextBlock, Versions
        from pydantic import AnyUrl
        
        # Create context block (requires exactly one source)
        context = {}
        if context_url:
            context["url"] = AnyUrl(context_url)
        elif inline_context:
            context["inline"] = inline_context
        elif ttl_url:
            context["derives_from"] = AnyUrl(ttl_url)
        else:
            return {"success": False, "error": "Must provide one of: context_url, inline_context, or ttl_url"}
        
        # Create the vocab entry
        try:
            entry = VocabEntry(
                prefix=prefix,
                uris={"primary": AnyUrl(uri), "alternates": []},
                context=ContextBlock(**context),
                versions=Versions(current="1.0", supported=["1.0"]),
                features=set(["temp_vocabulary"])
            )
            
            # Add to registry (this modifies the singleton instance)
            reg._v[prefix] = entry
            
            # Update alias map
            reg._alias[reg._norm(uri)] = prefix
            
            return {
                "success": True,
                "prefix": prefix,
                "uri": uri,
                "message": "Temporary vocabulary added to registry"
            }
        except Exception as e:
            return {
                "success": False,
                "error": f"Failed to add vocabulary: {str(e)}"
            }
    
    def detect_vocabularies(context_list: Union[List, Dict, str]) -> Dict[str, Any]:
        """
        Detect which vocabularies are used in a given context.
        
        Parameters:
            context_list: JSON-LD context to analyze (string, dict, or list)
            
        Returns:
            List of detected vocabulary prefixes with confidence scores
        """
        # Implementation of detect_vocabularies since it's missing from the registry module
        try:
            # This is a simplified implementation - we'll check if the input URI or context keys 
            # match any known vocabulary URIs in the registry
            detected = []
            
            # Handle different context formats
            if isinstance(context_list, str):
                # If it's just a URL string, check against registry URIs
                normalized_uri = reg._norm(context_list)
                if normalized_uri in reg._alias:
                    detected.append(reg._alias[normalized_uri])
                
            elif isinstance(context_list, dict):
                # If it's a JSON-LD context object, look through its keys
                context = context_list.get("@context", context_list)
                if isinstance(context, dict):
                    for prefix, uri in context.items():
                        if isinstance(uri, str) and not prefix.startswith("@"):
                            normalized_uri = reg._norm(uri)
                            if normalized_uri in reg._alias:
                                detected.append(reg._alias[normalized_uri])
                                
            elif isinstance(context_list, list):
                # If it's a list of context entries, process each one
                for item in context_list:
                    if isinstance(item, str):
                        normalized_uri = reg._norm(item)
                        if normalized_uri in reg._alias:
                            detected.append(reg._alias[normalized_uri])
                    elif isinstance(item, dict):
                        result = detect_vocabularies(item)
                        if result.get("success", False):
                            detected.extend(result.get("detected", {}).keys())
            
            # Remove duplicates
            detected = list(set(detected))
            
            # Add confidence scores (based on detection method)
            result = {}
            for prefix in detected:
                if prefix in reg._v:
                    entry = reg[prefix]
                    
                    # Determine confidence based on detection method
                    confidence = 0.7  # Default confidence
                    
                    # Check if it's an exact match to primary URI
                    if isinstance(context_list, str) and context_list == str(entry.uris.get("primary")):
                        confidence = 1.0
                    
                    result[prefix] = {
                        "prefix": prefix,
                        "uri": str(entry.uris.get("primary")),
                        "confidence": confidence
                    }
            
            return {
                "success": True,
                "detected": result,
                "count": len(result)
            }
        except Exception as e:
            return {
                "success": False,
                "error": f"Failed to detect vocabularies: {str(e)}"
            }
    
    # Add all functions to the tools dictionary
    tools["explore_registry"] = explore_registry
    tools["get_vocabulary_info"] = get_vocabulary_info
    tools["add_temp_vocabulary"] = add_temp_vocabulary
    tools["detect_vocabularies"] = detect_vocabularies
    
    return tools

# Let's test our registry tools

In [ ]:
reg_tools = registry_tools()

# Explore available vocabularies
reg_tools["explore_registry"]()

{'schema': {'prefix': 'schema',
  'uri': AnyUrl('https://schema.org/'),
  'versions': {'current': 'latest', 'supported': ['latest']},
  'features': ['inline_context',
   'supports_scoped_contexts',
   'uses_protection']},
 'croissant': {'prefix': 'croissant',
  'uri': AnyUrl('http://mlcommons.org/croissant/'),
  'versions': {'current': '1.0', 'supported': ['1.0']},
  'features': ['inline_context',
   'supports_scoped_contexts',
   'uses_protection']},
 'ro-crate': {'prefix': 'ro-crate',
  'uri': AnyUrl('https://w3id.org/ro/crate/1.2-DRAFT/context'),
  'versions': {'current': '1.2-DRAFT', 'supported': ['1.2-DRAFT']},
  'features': ['inline_context',
   'supports_scoped_contexts',
   'uses_protection']},
 'vc': {'prefix': 'vc',
  'uri': AnyUrl('https://www.w3.org/ns/credentials/v2'),
  'versions': {'current': 'v2', 'supported': ['v2']},
  'features': ['inline_context',
   'supports_scoped_contexts',
   'uses_protection']},
 'epcis': {'prefix': 'epcis',
  'uri': AnyUrl('https://ref.gs1.or

In [ ]:
# Get detailed info about a specific vocabulary
reg_tools["get_vocabulary_info"]("schema")

{'success': True,
 'prefix': 'schema',
 'uris': {'primary': 'https://schema.org/',
  'alternates': "[AnyUrl('http://schema.org/'), AnyUrl('https://schema.org/'), AnyUrl('http://schema.org/')]"},
 'context': {'url': 'https://schema.org/docs/jsonldcontext.jsonld',
  'inline': False,
  'derives_from': None,
  'sha256': '030762bb3ddb8eeaf44084c0978f70c36e59d6f2e9eaa57cc3fdca16049cdd44'},
 'versions': {'current': 'latest', 'supported': ['latest']},
 'features': ['inline_context', 'supports_scoped_contexts', 'uses_protection'],
 'tags': [],
 'strategy_defaults': {},
 'meta': {'publisher': 'Schema.org Community',
  'homepage': 'https://schema.org/'}}

In [ ]:
# Detect vocabularies in a context
reg_tools["detect_vocabularies"]("https://schema.org/")

{'success': True,
 'detected': {'schema': {'prefix': 'schema',
   'uri': 'https://schema.org/',
   'confidence': 1.0}},
 'count': 1}

# 2. Context Tools

Next, let's create tools for composing and analyzing JSON-LD contexts. These tools will wrap the functionality in `vocab/composer.py` and `vocab/collision.py`.

In [ ]:
#| export
def context_tools(composer=None, resolver=None) -> Dict[str, Callable]:
    """Create a set of tools for working with JSON-LD contexts.
    
    Returns:
        Dict mapping tool names to tool functions
    """
    from cogitarelink.vocab.composer import composer as default_composer
    from cogitarelink.vocab.collision import resolver as default_resolver
    
    comp = composer or default_composer
    res = resolver or default_resolver
    
    tools = {}
    
    def compose_context(prefixes: List[str], support_nest: bool = False, 
                      propagate: bool = True) -> Dict[str, Any]:
        """
        Compose a JSON-LD context from multiple vocabulary prefixes.
        
        Parameters:
            prefixes: List of vocabulary prefixes, ordered by priority
            support_nest: Whether to add @nest support
            propagate: Whether to allow context propagation
            
        Returns:
            Composed context object ready to use in a JSON-LD document
        """
        try:
            result = comp.compose(prefixes, support_nest, propagate)
            return {
                "success": True,
                "context": result
            }
        except Exception as e:
            return {
                "success": False,
                "error": str(e)
            }
    
    def analyze_context_compatibility(prefix_a: str, prefix_b: str) -> Dict[str, Any]:
        """
        Analyze compatibility between two vocabularies.
        
        Parameters:
            prefix_a: First vocabulary prefix
            prefix_b: Second vocabulary prefix
            
        Returns:
            Compatibility analysis with recommended strategy
        """
        from cogitarelink.vocab.collision import _prot_overlap
        
        try:
            # Get the resolution plan
            plan = res.choose(prefix_a, prefix_b)
            
            # Check for protected term overlap
            a_has, b_has, overlap = _prot_overlap(prefix_a, prefix_b)
            
            return {
                "success": True,
                "strategy": plan.strategy.value,
                "details": plan.details,
                "protected_terms": {
                    f"{prefix_a}_has_protected": a_has,
                    f"{prefix_b}_has_protected": b_has,
                    "has_overlap": overlap
                }
            }
        except Exception as e:
            return {
                "success": False,
                "error": f"Failed to analyze compatibility: {str(e)}"
            }
    
    def load_context_for_vocabulary(prefix: str) -> Dict[str, Any]:
        """
        Load the context for a specific vocabulary.
        
        Parameters:
            prefix: Vocabulary prefix
            
        Returns:
            JSON-LD context for the vocabulary
        """
        from cogitarelink.vocab.registry import registry
        
        try:
            entry = registry[prefix]
            context = entry.context_payload()
            
            return {
                "success": True,
                "prefix": prefix,
                "context": context,
                "source": "registry"
            }
        except Exception as e:
            return {
                "success": False,
                "error": f"Failed to load context: {str(e)}"
            }
    
    def apply_collision_strategy(data: Dict[str, Any], prefixes: List[str]) -> Dict[str, Any]:
        """
        Apply a collision strategy to a document with multiple vocabularies.
        
        Parameters:
            data: JSON-LD document to process
            prefixes: List of vocabulary prefixes to consider
            
        Returns:
            Processed document with collision strategy applied
        """
        from cogitarelink.vocab.collision import resolver
        
        if len(prefixes) < 2 or "@context" not in data:
            return {
                "success": True,
                "data": data,
                "message": "No changes needed"
            }
        
        try:
            # Get the plan for the first two vocabularies
            primary, secondary = prefixes[0], prefixes[1]
            plan = resolver.choose(primary, secondary)
            
            # Apply the strategy based on the plan
            from cogitarelink.vocab.composer import composer
            context = composer.compose(prefixes)
            
            # Create a new document with the composed context
            result = {**data}
            result["@context"] = context["@context"]
            
            return {
                "success": True,
                "data": result,
                "strategy": plan.strategy.value,
                "details": plan.details
            }
        except Exception as e:
            return {
                "success": False,
                "error": f"Failed to apply collision strategy: {str(e)}"
            }
    
    # Add all functions to the tools dictionary
    tools["compose_context"] = compose_context
    tools["analyze_context_compatibility"] = analyze_context_compatibility
    tools["load_context_for_vocabulary"] = load_context_for_vocabulary
    tools["apply_collision_strategy"] = apply_collision_strategy
    
    return tools

# Let's test our context tools

In [ ]:
ctx_tools = context_tools()

# Compose a context with multiple vocabularies
ctx_tools["compose_context"](prefixes=["schema", "dc"])

{'success': False,
 'error': "Client error '404 Not Found' for url 'https://www.dublincore.org/contexts/dc-terms/'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404"}

In [ ]:
# Analyze compatibility between two vocabularies
ctx_tools["analyze_context_compatibility"]("schema", "dc")

{'success': False,
 'error': "Failed to analyze compatibility: Client error '404 Not Found' for url 'https://www.dublincore.org/contexts/dc-terms/'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404"}

# 3. Retrieval Tools

Now, let's create tools for retrieving and processing vocabulary resources. These tools will wrap the functionality in `integration/retriever.py`.

In [ ]:
#| export
def retrieval_tools(retriever=None) -> Dict[str, Callable]:
    """Create a set of tools for retrieving and processing vocabulary resources.
    
    Returns:
        Dict mapping tool names to tool functions
    """
    from cogitarelink.integration.retriever import LODRetriever
    
    ret = retriever or LODRetriever()
    
    tools = {}
    
    def retrieve_vocabulary_resource(uri: str, format_hint: str = None) -> Dict[str, Any]:
        """
        Retrieve a vocabulary resource with automatic format detection.
        
        Parameters:
            uri: URI of the resource to retrieve
            format_hint: Optional hint about the expected format
            
        Returns:
            Retrieved resource converted to JSON-LD
        """
        result = ret.retrieve(uri)
        
        # Add format-specific metadata if successful
        if result.get("success", False):
            # Add information about the resource format
            if format_hint and format_hint != result.get("format"):
                result["format_requested"] = format_hint
        
        return result
    
    def extract_metadata(html_content: str, base_url: str = None, 
                       formats: List[str] = None) -> Dict[str, Any]:
        """
        Extract metadata from HTML in multiple formats using extruct.
        
        Parameters:
            html_content: HTML content to analyze
            base_url: Base URL for resolving relative URIs
            formats: List of formats to extract ('json-ld','rdfa','microdata','opengraph','microformat')
                    Default: all formats
            
        Returns:
            Dictionary with metadata for each format
        """
        from cogitarelink.integration.retriever import json_parse
        
        # Default to all supported formats if none specified
        formats = formats or ['json-ld', 'rdfa', 'microdata', 'opengraph', 'microformat']
        
        try:
            # Use extruct library to extract metadata
            import extruct
            from w3lib.html import get_base_url
            
            # Set the base URL for resolving relative URIs
            base = get_base_url(html_content, base_url or '')
            
            # Extract all requested metadata formats
            extracted = extruct.extract(
                html_content,
                base_url=base,
                syntaxes=formats,
                uniform=True  # Return uniform results
            )
            
            # Process and validate each format
            processed_data = {}
            format_counts = {}
            
            for format_type, items in extracted.items():
                if not items:
                    continue
                    
                if format_type == 'json-ld':
                    # Process and validate each JSON-LD item
                    valid_items = []
                    for item in items:
                        json_ld, error = json_parse(json.dumps(item), uri=base)
                        if json_ld:
                            valid_items.append(json_ld)
                    
                    if valid_items:
                        processed_data[format_type] = valid_items
                        format_counts[format_type] = len(valid_items)
                else:
                    # Other formats don't need validation
                    processed_data[format_type] = items
                    format_counts[format_type] = len(items)
            
            # Check if we found any metadata
            if not processed_data:
                return {
                    "success": False,
                    "error": f"No metadata found in formats: {', '.join(formats)}",
                    "formats_requested": formats
                }
            
            return {
                "success": True,
                "metadata": processed_data,
                "formats": list(processed_data.keys()),
                "counts": format_counts,
                "total_items": sum(format_counts.values())
            }
            
        except ImportError:
            # Fallback to limited BeautifulSoup-based extraction if extruct is not available
            from bs4 import BeautifulSoup
            
            results = {}
            counts = {}
            
            try:
                soup = BeautifulSoup(html_content, "html.parser")
                
                # Only JSON-LD is supported with BeautifulSoup fallback
                if 'json-ld' in formats:
                    jsonld_scripts = soup.select("script[type='application/ld+json']")
                    if jsonld_scripts:
                        jsonld_data = []
                        for script in jsonld_scripts:
                            json_ld, error = json_parse(script.string, uri=base_url)
                            if json_ld:
                                jsonld_data.append(json_ld)
                        
                        if jsonld_data:
                            results['json-ld'] = jsonld_data
                            counts['json-ld'] = len(jsonld_data)
                
                # OpenGraph meta tags
                if 'opengraph' in formats:
                    og_data = {}
                    og_tags = soup.select("meta[property^='og:']")
                    for tag in og_tags:
                        prop = tag.get('property')
                        content = tag.get('content')
                        if prop and content:
                            og_data[prop] = content
                    
                    if og_data:
                        results['opengraph'] = [og_data]  # Match uniform format with a list
                        counts['opengraph'] = 1
                
                if not results:
                    return {
                        "success": False,
                        "error": f"No metadata found with BeautifulSoup fallback in formats: {', '.join(formats)}",
                        "formats_requested": formats,
                        "note": "Limited extraction capabilities without extruct"
                    }
                
                return {
                    "success": True,
                    "metadata": results,
                    "formats": list(results.keys()),
                    "counts": counts,
                    "total_items": sum(counts.values()),
                    "note": "Limited extraction with BeautifulSoup fallback"
                }
                
            except Exception as e:
                return {
                    "success": False,
                    "error": f"Metadata extraction error: {str(e)}",
                    "formats_requested": formats
                }
        except Exception as e:
            return {
                "success": False,
                "error": f"Metadata extraction error: {str(e)}",
                "formats_requested": formats
            }
    
    def extract_embedded_jsonld(html_content: str, base_url: str = None) -> Dict[str, Any]:
        """
        Extract JSON-LD content embedded in HTML.
        
        Parameters:
            html_content: HTML content to analyze
            base_url: Base URL for resolving relative URIs
            
        Returns:
            Extracted JSON-LD data if found
        """
        # Use the more comprehensive extract_metadata function with 'json-ld' format only
        result = extract_metadata(html_content, base_url, formats=['json-ld'])
        
        if result.get("success", False):
            return {
                "success": True,
                "data": result.get("metadata", {}).get("json-ld", []),
                "count": result.get("counts", {}).get("json-ld", 0),
                "error": None
            }
        else:
            return {
                "success": False,
                "error": result.get("error", "JSON-LD extraction failed")
            }
    
    def convert_format(content: str, from_format: str, to_format: str = "json-ld") -> Dict[str, Any]:
        """
        Convert between different RDF serialization formats.
        
        Parameters:
            content: Content to convert
            from_format: Source format (turtle, xml, n3, etc.)
            to_format: Target format (defaults to json-ld)
            
        Returns:
            Converted content
        """
        try:
            from rdflib import Graph
            
            # Parse the input format
            g = Graph()
            g.parse(data=content, format=from_format)
            
            # Serialize to the target format
            if to_format == "json-ld":
                result = json.loads(g.serialize(format="json-ld"))
                return {
                    "success": True,
                    "data": result,
                    "from_format": from_format,
                    "to_format": to_format,
                    "triples_count": len(g)
                }
            else:
                result = g.serialize(format=to_format)
                return {
                    "success": True,
                    "data": result,
                    "from_format": from_format,
                    "to_format": to_format,
                    "triples_count": len(g)
                }
        except Exception as e:
            return {
                "success": False,
                "error": f"Conversion error: {str(e)}"
            }
    
    def wikidata_search(query: str, limit: int = 10, language: str = "en") -> Dict[str, Any]:
        """
        Search for entities in Wikidata matching a query.
        
        Parameters:
            query: Search text
            limit: Maximum number of results
            language: Language code for labels
            
        Returns:
            List of matching Wikidata entities
        """
        from cogitarelink.integration.retriever import search_wikidata
        
        results = search_wikidata(query, limit, language)
        
        return {
            "success": not any("error" in item for item in results),
            "results": results,
            "count": len(results),
            "query": query
        }
    
    def wikidata_entity_details(entity_id: str) -> Dict[str, Any]:
        """
        Get detailed information about a Wikidata entity.
        
        Parameters:
            entity_id: Wikidata entity ID (Q-number)
            
        Returns:
            Detailed entity information
        """
        return ret.get_entity_details(entity_id)
    
    def align_property_to_wikidata(property_name: str, 
                                  description: str = None) -> Dict[str, Any]:
        """
        Find the closest matching Wikidata property for a dataset field.
        
        Parameters:
            property_name: Name of the property to align
            description: Optional property description
            
        Returns:
            Wikidata alignment suggestions
        """
        from cogitarelink.integration.retriever import search_wikidata
        
        # Search for property by name
        query = property_name
        if description:
            # Add first sentence of description to improve search
            first_sentence = description.split('.')[0] if '.' in description else description
            query = f"{property_name} {first_sentence}"
        
        results = search_wikidata(query, limit=5)
        
        # Basic confidence scoring
        scored_results = []
        for item in results:
            if "error" in item:
                continue
                
            # Simple confidence scoring based on label similarity
            confidence = 0.0
            if item.get("label", "").lower() == property_name.lower():
                confidence = 0.9
            elif property_name.lower() in item.get("label", "").lower():
                confidence = 0.7
            elif any(property_name.lower() in word.lower() for word in item.get("label", "").split()):
                confidence = 0.5
            else:
                confidence = 0.3
                
            # Boost confidence if description matches
            if description and description.lower() in item.get("description", "").lower():
                confidence = min(confidence + 0.2, 1.0)
                
            item["confidence"] = round(confidence, 2)
            scored_results.append(item)
        
        # Sort by confidence
        scored_results.sort(key=lambda x: x.get("confidence", 0), reverse=True)
        
        return {
            "success": True,
            "property": property_name,
            "matches": scored_results,
            "best_match": scored_results[0] if scored_results else None
        }
    
    # Add all functions to the tools dictionary
    tools["retrieve_vocabulary_resource"] = retrieve_vocabulary_resource
    tools["extract_metadata"] = extract_metadata
    tools["extract_embedded_jsonld"] = extract_embedded_jsonld
    tools["convert_format"] = convert_format
    tools["wikidata_search"] = wikidata_search
    tools["wikidata_entity_details"] = wikidata_entity_details
    tools["align_property_to_wikidata"] = align_property_to_wikidata
    
    return tools

# Let's test our retrieval tools with a Wikidata search

In [ ]:
ret_tools = retrieval_tools()

# Search Wikidata for a term
ret_tools["wikidata_search"]("machine learning", limit=3)

{'success': True,
 'results': [{'id': 'Q2539',
   'uri': 'http://www.wikidata.org/entity/Q2539',
   'label': 'machine learning',
   'description': 'scientific study of algorithms and statistical models that computer systems use to perform tasks without explicit instructions',
   'url': '//www.wikidata.org/wiki/Q2539'},
  {'id': 'Q6723676',
   'uri': 'http://www.wikidata.org/entity/Q6723676',
   'label': 'Machine Learning',
   'description': 'journal',
   'url': '//www.wikidata.org/wiki/Q6723676'},
  {'id': 'Q108371168',
   'uri': 'http://www.wikidata.org/entity/Q108371168',
   'label': 'Machine Learning',
   'description': 'video game series',
   'url': '//www.wikidata.org/wiki/Q108371168'}],
 'count': 3,
 'query': 'machine learning'}

In [ ]:
# Let's test our multi-format metadata extraction
html_example = """
<!DOCTYPE html>
<html>
<head>
    <title>Product Page Example</title>
    <!-- OpenGraph metadata -->
    <meta property="og:title" content="Sample Product" />
    <meta property="og:type" content="product" />
    <meta property="og:url" content="https://example.com/product/123" />
    <meta property="og:image" content="https://example.com/product/123.jpg" />
    
    <!-- JSON-LD metadata -->
    <script type="application/ld+json">
    {
        "@context": "https://schema.org/",
        "@type": "Product",
        "name": "Sample Product",
        "description": "This is a sample product for testing",
        "sku": "SAMPLE123",
        "brand": {
            "@type": "Brand",
            "name": "Test Brand"
        },
        "offers": {
            "@type": "Offer",
            "price": 19.99,
            "priceCurrency": "USD",
            "availability": "https://schema.org/InStock"
        }
    }
    </script>
    
    <!-- RDFa metadata -->
    <div vocab="https://schema.org/" typeof="Organization">
        <span property="name">Example Company</span>
        <div property="address" typeof="PostalAddress">
            <span property="streetAddress">123 Main St</span>
            <span property="addressLocality">Anytown</span>
        </div>
    </div>
</head>
<body>
    <h1>Sample Product</h1>
    <p>This is a sample product for testing metadata extraction.</p>
</body>
</html>
"""

# Extract all formats
multi_result = ret_tools["extract_metadata"](html_content=html_example, base_url="https://example.com/")
print(f"Found {multi_result.get('total_items', 0)} metadata items in {multi_result.get('formats', [])} formats")
multi_result.get('counts', {})

Found 2 metadata items in ['json-ld', 'opengraph'] formats


{'json-ld': 1, 'opengraph': 1}

In [ ]:
#| export
class VocabToolAgent(Agent):
    """Agent with specialized tools for working with semantic vocabularies."""
    
    def __init__(self, name: str = "vocab-agent"):
        super().__init__(name=name)
        self._register_vocab_tools()
        
    def run_tool(self, tool_name: str, **kwargs) -> Any:
        """Override run_tool to fix name parameter issue."""
        try:
            # Get the tool directly from the registry
            tool = self.tools.get(tool_name)
            func = tool["function"]
            # Execute the function directly
            result = func(**kwargs)
            self.context.log_action(tool_name, kwargs, result)
            return result
        except Exception as e:
            self.context.log_action(tool_name, kwargs, e)
            raise
    
    def _register_vocab_tools(self):
        """Register all vocabulary-specific tools."""
        # Register registry tools
        for name, func in registry_tools().items():
            self.register_tool(func=func, name=name)
        
        # Register context tools
        for name, func in context_tools().items():
            self.register_tool(func=func, name=name)
        
        # Register retrieval tools
        for name, func in retrieval_tools().items():
            self.register_tool(func=func, name=name)
        
        # Register specialized vocabulary tools for Croissant
        self._register_croissant_tools()
    
    def _register_croissant_tools(self):
        """Register specialized tools for working with Croissant schemas."""
        
        # Define the create_croissant_dataset function
        def create_croissant_dataset(name: str, description: str = None, 
                                   recordsets: List[Dict] = None) -> Dict[str, Any]:
            """
            Create a Croissant dataset with the specified properties.
            
            Parameters:
                name: Dataset name
                description: Optional dataset description
                recordsets: Optional list of recordset definitions
                
            Returns:
                Complete Croissant dataset object
            """
            # Try to get the Croissant context
            try:
                # Use tool function directly to avoid run_tool parameter issue
                tool = self.tools.get("load_context_for_vocabulary")
                ctx_result = tool["function"](prefix="croissant")
                
                if not ctx_result.get("success", False):
                    # Try to load from TTL (fallback)
                    ttl_url = "https://raw.githubusercontent.com/mlcommons/croissant/main/docs/croissant.ttl"
                    
                    # Use tool functions directly to avoid run_tool parameter issue
                    retrieve_tool = self.tools.get("retrieve_vocabulary_resource")
                    ttl_result = retrieve_tool["function"](uri=ttl_url, format_hint="turtle")
                    
                    if ttl_result.get("success", False) and "content" in ttl_result:
                        # Convert TTL to JSON-LD
                        convert_tool = self.tools.get("convert_format")
                        convert_result = convert_tool["function"](
                            content=ttl_result["content"],
                            from_format="turtle"
                        )
                        
                        if convert_result.get("success", False):
                            # Add as temporary vocabulary
                            add_vocab_tool = self.tools.get("add_temp_vocabulary")
                            add_vocab_tool["function"](
                                prefix="croissant",
                                uri="http://mlcommons.org/croissant/",
                                inline_context=convert_result["data"].get("@context", {})
                            )
                            
                            # Try to load again
                            ctx_result = tool["function"](prefix="croissant")
            except Exception as e:
                self.context.remember("croissant_error", str(e))
                # Use a minimal context as fallback
                ctx_result = {
                    "success": True,
                    "context": {
                        "@context": {
                            "@vocab": "http://mlcommons.org/croissant/"
                        }
                    }
                }
            
            # Create the dataset
            dataset = {
                "@context": ctx_result.get("context", {}).get("@context", 
                                                           {"@vocab": "http://mlcommons.org/croissant/"}),
                "@type": "Dataset",
                "name": name
            }
            
            # Add optional properties
            if description:
                dataset["description"] = description
                
            # Add recordsets if provided
            if recordsets:
                dataset["recordSet"] = recordsets
            
            return {
                "success": True,
                "dataset": dataset
            }
        
        # Register the function properly
        self.register_tool(func=create_croissant_dataset, name="create_croissant_dataset")
        
        # Define the align_dataset_to_wikidata function
        def align_dataset_to_wikidata(dataset: Dict[str, Any],
                                    confidence_threshold: float = 0.7) -> Dict[str, Any]:
            """
            Align a Croissant dataset fields with Wikidata concepts.
            
            Parameters:
                dataset: Croissant dataset to align
                confidence_threshold: Minimum confidence for automatic alignment
                
            Returns:
                Dataset with Wikidata alignments added
            """
            # Check if this is a Croissant dataset
            if not isinstance(dataset, dict) or "@type" not in dataset or dataset.get("@type") != "Dataset":
                return {
                    "success": False,
                    "error": "Input must be a Croissant dataset object"
                }
            
            # Create a copy to modify
            result = dataset.copy()
            alignments = {}
            
            # Process recordsets
            recordsets = result.get("recordSet", [])
            if not isinstance(recordsets, list):
                recordsets = [recordsets]
                
            for i, recordset in enumerate(recordsets):
                fields = recordset.get("field", [])
                if not isinstance(fields, list):
                    fields = [fields]
                
                for j, field in enumerate(fields):
                    # Skip fields that already have alignments
                    if "sameAs" in field:
                        continue
                        
                    # Get field metadata
                    field_name = field.get("name", "")
                    field_desc = field.get("description", "")
                    
                    # Try to align with Wikidata
                    try:
                        # Use tool function directly to avoid run_tool parameter issue
                        align_tool = self.tools.get("align_property_to_wikidata")
                        alignment = align_tool["function"](
                            property_name=field_name,
                            description=field_desc
                        )
                        
                        if alignment.get("success", False) and alignment.get("best_match"):
                            best_match = alignment["best_match"]
                            
                            # Store alignment info
                            alignments[field_name] = {
                                "wikidata_id": best_match.get("id"),
                                "wikidata_uri": best_match.get("uri"),
                                "label": best_match.get("label"),
                                "confidence": best_match.get("confidence")
                            }
                            
                            # Add Wikidata reference if confidence is high enough
                            if best_match.get("confidence", 0) >= confidence_threshold:
                                # Modify the field in place
                                fields[j]["sameAs"] = best_match.get("uri")
                    except Exception as e:
                        # Just log the error and continue
                        self.context.remember(f"alignment_error_{field_name}", str(e))
                
                # Update the fields in the recordset
                recordsets[i]["field"] = fields
            
            # Update the recordsets in the result
            if len(recordsets) == 1:
                result["recordSet"] = recordsets[0]
            else:
                result["recordSet"] = recordsets
            
            return {
                "success": True,
                "dataset": result,
                "alignments": alignments,
                "alignment_count": len(alignments)
            }
        
        # Register the function properly
        self.register_tool(func=align_dataset_to_wikidata, name="align_dataset_to_wikidata")
    
    def resolve_croissant_vocabulary(self) -> Dict[str, Any]:
        """Robust resolution of the Croissant vocabulary."""
        # Use the override run_tool method to fix parameter issue
        vocab_result = self.run_tool("get_vocabulary_info", prefix="croissant")
        
        if not vocab_result.get("success", False):
            # Try to resolve using the TTL file
            ttl_resource = self.run_tool("retrieve_vocabulary_resource",
                uri="https://raw.githubusercontent.com/mlcommons/croissant/main/docs/croissant.ttl",
                format_hint="turtle"
            )
            
            if ttl_resource.get("success", False) and "content" in ttl_resource:
                # Convert TTL to JSON-LD
                convert_result = self.run_tool("convert_format",
                                             content=ttl_resource["content"],
                                             from_format="turtle")
                
                if convert_result.get("success", False):
                    # Add as temporary vocabulary
                    temp_result = self.run_tool("add_temp_vocabulary",
                                              prefix="croissant",
                                              uri="http://mlcommons.org/croissant/",
                                              inline_context=convert_result["data"].get("@context", {}))
                    
                    if temp_result.get("success", False):
                        # Try to get info again
                        vocab_result = self.run_tool("get_vocabulary_info", prefix="croissant")
        
        return vocab_result

# Let's test our VocabToolAgent

In [ ]:
vocab_agent = VocabToolAgent(name="test-vocab-agent")

# List all available tools
vocab_agent.run_tool("list_tools")

[{'name': 'get_version',
  'description': 'Get the CogitareLink version',
  'signature': {}},
 {'name': 'list_tools',
  'description': 'List all available tools',
  'signature': {}},
 {'name': 'get_agent_memory',
  'description': 'Get all items in agent memory',
  'signature': {}},
 {'name': 'explore_registry',
  'description': '\n        Explore available vocabularies in the registry with optional filtering.\n\n        Parameters:\n            filter_tags: Optional tags to filter vocabularies\n\n        Returns:\n            Dictionary of vocabulary entries matching the filter\n        ',
  'signature': {'filter_tags': 'List[str]'}},
 {'name': 'get_vocabulary_info',
  'description': '\n        Get detailed information about a specific vocabulary.\n\n        Parameters:\n            prefix: Vocabulary prefix\n\n        Returns:\n            Complete vocabulary entry information\n        ',
  'signature': {'prefix': 'str'}},
 {'name': 'add_temp_vocabulary',
  'description': '\n        A

In [ ]:
# Create a Croissant dataset
croissant_dataset = vocab_agent.run_tool("create_croissant_dataset", 
                                        name="Weather Measurements 2023",
                                        description="Daily temperature and precipitation measurements for 2023",
                                        recordsets=[{
                                            "@type": "RecordSet",
                                            "name": "weather_data",
                                            "field": [
                                                {"@type": "Property", "name": "date", "dataType": "date", "description": "Measurement date"},
                                                {"@type": "Property", "name": "temperature", "dataType": "float", "description": "Temperature in Celsius"},
                                                {"@type": "Property", "name": "precipitation", "dataType": "float", "description": "Rainfall in millimeters"}
                                            ]
                                        }])

In [ ]:
croissant_dataset

{'success': True,
 'dataset': {'@context': {'@vocab': 'http://mlcommons.org/croissant/'},
  '@type': 'Dataset',
  'name': 'Weather Measurements 2023',
  'description': 'Daily temperature and precipitation measurements for 2023',
  'recordSet': [{'@type': 'RecordSet',
    'name': 'weather_data',
    'field': [{'@type': 'Property',
      'name': 'date',
      'dataType': 'date',
      'description': 'Measurement date'},
     {'@type': 'Property',
      'name': 'temperature',
      'dataType': 'float',
      'description': 'Temperature in Celsius'},
     {'@type': 'Property',
      'name': 'precipitation',
      'dataType': 'float',
      'description': 'Rainfall in millimeters'}]}]}}

In [ ]:
# Align dataset with Wikidata
aligned_dataset = vocab_agent.run_tool("align_dataset_to_wikidata", 
                                     dataset=croissant_dataset.get("dataset", {}),
                                     confidence_threshold=0.6)

In [ ]:
# Show alignments
aligned_dataset.get("alignments", {})

{}

# Conclusion

In this notebook, we've built a set of vocabulary tools that extend our agent infrastructure:

1. **Registry Tools**: For exploring, querying, and manipulating the vocabulary registry
2. **Context Tools**: For composing, analyzing, and working with JSON-LD contexts
3. **Retrieval Tools**: For fetching, converting, and processing semantic data
4. **Croissant Tools**: Specialized tools for working with Croissant datasets

These tools are bundled into a `VocabToolAgent` class that makes them easily accessible and handles error cases gracefully. The agent includes robust fallback mechanisms for vocabulary resolution, ensuring it can work even when primary sources are unavailable.

Our implementation leverages CogitareLink's existing systems for vocabulary management while providing a clean, tool-based interface that follows the design principles of our agent infrastructure.

In the next notebook, we can explore how to use this agent for more complex data workflows, such as:

1. Multi-vocabulary dataset creation
2. Schema alignment with external ontologies
3. Automated metadata extraction and enrichment
4. Integration with Jupyter notebooks for interactive data exploration

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()